# Лабораторная работа №3

### Генерация текста с помощью LSTM

**Цель работы:** Создать сеть на базе LSTM используя TensorFlow (Keras). Сеть должна принимать на вход текстовый файл и на его базе генерировать свою абракадабру.

**Задачи:**
1. Подготовить текстовый файл для обучения
2. Создать LSTM модель для генерации текста
3. Обучить модель на подготовленном тексте
4. Сгенерировать новый текст (абракадабру) на основе обученной модели
5. Проанализировать результаты генерации

## 1. Импорт библиотек

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
import random

# Для воспроизводимости результатов
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

print(f"TensorFlow version: {tf.__version__}")

TensorFlow version: 2.21.0


## 2. Подготовка обучающего текста

In [2]:
# Обучающий текст
sample_text = """Летом я поеду на море купаться и загорать.
Я возьму с собой полотенце очки и крем от загара.
На пляже я буду строить замки из белого песка.
В море я увижу медуз ракушек и маленьких крабов.
Вечером мы отправимся гулять по набережной.
В кафе я закажу сок пиццу и вкусный чизкейк.
Из отпуска я привезу сувениры и много красивых фото.
"""

# Повтор 20 раз
training_text = sample_text * 20

# Сохраняем в файл
with open("train_text.txt", "w", encoding="utf-8") as f:
    f.write(training_text)

print(f"Размер обучающего текста: {len(training_text)} символов")
print(f"\nПервые 200 символов текста:")
print(training_text[:200])

Размер обучающего текста: 6620 символов

Первые 200 символов текста:
Летом я поеду на море купаться и загорать.
Я возьму с собой полотенце очки и крем от загара.
На пляже я буду строить замки из белого песка.
В море я увижу медуз ракушек и маленьких крабов.
Вечером мы 


## 3. Загрузка и предобработка текста

In [3]:
# Загрузка текста из файла
with open('train_text.txt', 'r', encoding='utf-8') as f:
    text = f.read().lower()  # приводим к нижнему регистру

print(f"Общая длина текста: {len(text)} символов")

# Получение уникальных символов
chars = sorted(list(set(text)))
char_to_idx = {char: idx for idx, char in enumerate(chars)}
idx_to_char = {idx: char for idx, char in enumerate(chars)}

print(f"Количество уникальных символов: {len(chars)}")
print(f"Уникальные символы: {''.join(chars)}")

Общая длина текста: 6620 символов
Количество уникальных символов: 31
Уникальные символы: 
 .абвгдежзийклмнопрстуфхцчшыья


## 4. Создание обучающих последовательностей

In [4]:
# Параметры
seq_length = 40      # длина входной последовательности
step = 3             # шаг сдвига

# Создание последовательностей
sentences = []       # входные последовательности
next_chars = []      # целевые символы (то, что нужно предсказать)

for i in range(0, len(text) - seq_length, step):
    sentences.append(text[i:i + seq_length])
    next_chars.append(text[i + seq_length])

print(f"Количество обучающих примеров: {len(sentences)}")
print(f"\nПример последовательности:")
print(f"Вход: {sentences[0]}")
print(f"Цель: {next_chars[0]}")

Количество обучающих примеров: 2194

Пример последовательности:
Вход: летом я поеду на море купаться и загорат
Цель: ь


## 5. One-hot кодирование

In [5]:
# Создание массивов для обучения
X = np.zeros((len(sentences), seq_length, len(chars)), dtype=bool)
y = np.zeros((len(sentences), len(chars)), dtype=bool)

# Заполнение массивов
for i, sentence in enumerate(sentences):
    for t, char in enumerate(sentence):
        X[i, t, char_to_idx[char]] = 1
    y[i, char_to_idx[next_chars[i]]] = 1

print(f"Форма X (примеры, длина, символы): {X.shape}")
print(f"Форма y (примеры, символы): {y.shape}")

Форма X (примеры, длина, символы): (2194, 40, 31)
Форма y (примеры, символы): (2194, 31)


## 6. Создание LSTM модели


In [6]:
# Создание модели
model = Sequential([
    LSTM(128, input_shape=(seq_length, len(chars))),  # LSTM слой на 128 нейронов
    Dense(len(chars), activation='softmax')            # Выходной слой с softmax
])

# Компиляция модели
model.compile(
    loss='categorical_crossentropy',  # функция потерь
    optimizer='adam'                   # оптимизатор
)

# Вывод структуры модели
model.summary()

c:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 128)            │        81,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 31)             │         3,999 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 85,919 (335.62 KB)

 Trainable params: 85,919 (335.62 KB)

 Non-trainable params: 0 (0.00 B)

## 7. Обучение модели

In [7]:
print("\nНачало обучения модели...")
print("-" * 40)

history = model.fit(
    X, y,
    batch_size=64,      # размер батча
    epochs=27,          # количество эпох
    verbose=1           # вывод прогресса
)

print("-" * 40)
print("Обучение завершено!")

# Вывод финальной ошибки
final_loss = history.history['loss'][-1]
print(f"Финальная loss: {final_loss:.6f}")


Начало обучения модели...
----------------------------------------
Epoch 1/27
35/35 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - loss: 3.2064
Epoch 2/27
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 3.0795
Epoch 3/27
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 3.0226
Epoch 4/27
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 2.9199
Epoch 5/27
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 2.7795
Epoch 6/27
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 2.6392
Epoch 7/27
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 2.4350
Epoch 8/27
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 2.2492
Epoch 9/27
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 2.0605
Epoch 10/27
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 1.8595
Epoch 11/27
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 1.6826
Epoch 12/27
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 1.5019
Epoch 13/27
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 1.3593
Epoch 14/27
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 1.2107
Epo

## 8. Генерация текста (абракадабры)

In [8]:
# Выбор случайного начального фрагмента текста (seed)
start_idx = random.randint(0, len(text) - seq_length - 1)
seed_text = text[start_idx:start_idx + seq_length]
seed_text_copy = seed_text
generated = seed_text

print("=" * 60)
print("ГЕНЕРАЦИЯ АБРАКАДАБРЫ")
print("=" * 60)
print(f"\nНачальная строка (seed):")
print(f"\"{seed_text}\"")
print()
print("СГЕНЕРИРОВАННЫЙ ТЕКСТ:")

# Генерация 300 символов
for i in range(300):
    # Подготовка входных данных
    x_pred = np.zeros((1, seq_length, len(chars)))
    for t, char in enumerate(seed_text):
        if char in char_to_idx:
            x_pred[0, t, char_to_idx[char]] = 1.0
    
    # Предсказание следующего символа
    preds = model.predict(x_pred, verbose=0)[0]
    next_idx = np.argmax(preds)      # выбираем символ с максимальной вероятностью
    next_char = idx_to_char[next_idx]
    
    # Обновление строки
    generated += next_char
    seed_text = seed_text[1:] + next_char

print(generated)

ГЕНЕРАЦИЯ АБРАКАДАБРЫ

Начальная строка (seed):
"й чизкейк.
из отпуска я привезу сувениры"

СГЕНЕРИРОВАННЫЙ ТЕКСТ:
й чизкейк.
из отпуска я привезу сувениры и много красивх офото.
летом я поеду на море кспатьс  и загорать.
я везьууус сссооооойтцнццце ои и  кре  о заааааан.
и  гоуеууусусс рротть  миммиз гллеек   ска.ввее мо оротя яууу уудуроеик рое ааьено ооки вививх ыы оотойототллеееокк к ои фафаясппппаниии  гоббввсввви и и некойййшшйй оллеекек  и ккаф


## 9. Сохранение результатов

In [9]:
# Сохранение сгенерированного текста в файл
with open("generated_abracadabra.txt", "w", encoding="utf-8") as f:
    f.write("=" * 60 + "\n")
    f.write("СГЕНЕРИРОВАННЫЙ ТЕКСТ (АБРАКАДАБРА)\n")
    f.write("=" * 60 + "\n\n")
    f.write(f"Начальная строка: {seed_text_copy}\n\n")
    f.write(generated)
    f.write("\n\n" + "=" * 60 + "\n")
    f.write(f"Всего сгенерировано символов: {len(generated)}\n")

print("\nРезультат сохранен в файл: generated_abracadabra.txt")


Результат сохранен в файл: generated_abracadabra.txt


## 10. Выводы

In [10]:
print(f"\nСтатистика:")
print(f"Обучающий текст: {len(training_text)} символов")
print(f"Уникальных символов: {len(chars)}")
print(f"Обучающих примеров: {len(sentences)}")
print(f"Финальная loss: {final_loss:.6f}")
print(f"Сгенерировано символов: {len(generated)}")


Статистика:
Обучающий текст: 6620 символов
Уникальных символов: 31
Обучающих примеров: 2194
Финальная loss: 0.154312
Сгенерировано символов: 340


**Анализ результатов:**
1. LSTM сеть успешно обучилась на предоставленном тексте и научилась предсказывать следующий символ
2. Сгенерированный текст содержит слова и фразы из обучающего текста, но с повторениями букв и пробелов — это и есть "абракадабра"
3. При многократном повторении одного текста модель хорошо его запоминает

**Вывод:** LSTM успешно решает задачу генерации текста. Модель принимает текстовый файл на вход и генерирует на его основе новый текст.